In [2]:
from pathlib import Path
import zipfile
import pandas as pd

ZIP_PATH = Path("../data/raw/archive.zip")

print("Dataset exists:", ZIP_PATH.exists())
print("Size (MB):", round(ZIP_PATH.stat().st_size / (1024 * 1024), 2))

Dataset exists: True
Size (MB): 168.58


In [3]:
with zipfile.ZipFile(ZIP_PATH, "r") as z:
    files = z.namelist()

print("Files inside ZIP:")
for f in files:
    print(f)

Files inside ZIP:
sample.csv
twcs/twcs.csv


In [4]:
with zipfile.ZipFile(ZIP_PATH, "r") as z:
    with z.open("twcs/twcs.csv") as f:
        df_sample = pd.read_csv(f, nrows=5)

print("Columns:")
print(df_sample.columns.tolist())

display(df_sample)

Columns:
['tweet_id', 'author_id', 'inbound', 'created_at', 'text', 'response_tweet_id', 'in_response_to_tweet_id']


,tweet_id,author_id,inbound,created_at,text,response_tweet_id,in_response_to_tweet_id
0,1,sprintcare,False,Tue Oct 31 22:10:47 +0000 2017,@115712 I understand. I would like to assist y...,2.0,3
1,2,115712,True,Tue Oct 31 22:11:45 +0000 2017,@sprintcare and how do you propose we do that,NaN,1
2,3,115712,True,Tue Oct 31 22:08:27 +0000 2017,@sprintcare I have sent several private messag...,1.0,4
3,4,sprintcare,False,Tue Oct 31 21:54:49 +0000 2017,@115712 Please send us a Private Message so th...,3.0,5
4,5,115712,True,Tue Oct 31 21:49:35 +0000 2017,@sprintcare I did.,4.0,6


In [5]:
total_rows = 0
inbound_count = 0
outbound_count = 0

with zipfile.ZipFile(ZIP_PATH, "r") as z:
    with z.open("twcs/twcs.csv") as f:
        for chunk in pd.read_csv(f, chunksize=200_000):
            total_rows += len(chunk)
            inbound_count += chunk["inbound"].sum()
            outbound_count += (~chunk["inbound"]).sum()

print("Total tweets:", total_rows)
print("Customer tweets (inbound):", inbound_count)
print("Brand tweets (outbound):", outbound_count)

Total tweets: 2811774
Customer tweets (inbound): 1537843
Brand tweets (outbound): 1273931


In [6]:
brand_counts = {}

with zipfile.ZipFile(ZIP_PATH, "r") as z:
    with z.open("twcs/twcs.csv") as f:
        for chunk in pd.read_csv(f, chunksize=200_000):
            outbound = chunk[chunk["inbound"] == False]
            counts = outbound["author_id"].value_counts()

            for brand, count in counts.items():
                brand_counts[brand] = brand_counts.get(brand, 0) + count

top_brands = (
    pd.Series(brand_counts)
    .sort_values(ascending=False)
    .head(20)
)

print(top_brands)

AmazonHelp         169840
AppleSupport       106860
Uber_Support        56270
SpotifyCares        43265
Delta               42253
Tesco               38573
AmericanAir         36764
TMobileHelp         34317
comcastcares        33031
British_Airways     29361
SouthwestAir        28977
VirginTrains        27817
Ask_Spectrum        25860
XboxSupport         24557
sprintcare          22381
hulu_support        21872
sainsburys          19466
GWRHelp             19364
AskPlayStation      19098
ChipotleTweets      18749
dtype: int64


In [7]:
amazon_samples = []

with zipfile.ZipFile(ZIP_PATH, "r") as z:
    with z.open("twcs/twcs.csv") as f:
        for chunk in pd.read_csv(f, chunksize=200_000):
            rows = chunk[
                (chunk["author_id"] == "AmazonHelp") &
                (chunk["inbound"] == False) &
                (chunk["in_response_to_tweet_id"].notna())
            ]

            if len(rows) > 0:
                amazon_samples.append(
                    rows.sample(min(10, len(rows)), random_state=42)
                )

amazon_samples = pd.concat(amazon_samples).head(20)

display(
    amazon_samples[
        ["tweet_id", "text", "in_response_to_tweet_id"]
    ]
)

,tweet_id,text,in_response_to_tweet_id
111707,139073,@147339 Seems like you had an unpleasant exper...,139074.0
193804,227948,@165000 Sorry but we can't access your account...,206938.0
15366,20033,@120404 please check this link here: https://t...,20034.0
150925,180948,"@158504 I'm sorry for any trouble, being a soc...",180947.0
190765,224596,@169659 アマゾンです。Amazonプライムは様々な特典をご用意させていただいておりま...,224597.0
94497,120225,@142689 どうぞよろしくお願いいたします。\nまた何かご不明な点がございましたらお知ら...,120223.0
94422,120148,@142675 Thanks for reaching out to us! You can...,120150.0
91299,115817,"@141789 Olá, Amanda! Conte para nós, quais liv...",115818.0
53205,65629,"@131026 Hey, this being a social platform we'l...",65628.0
149509,179472,@158274 Have you been in touch with us here: ...,179473.0


In [8]:
reply_ids = set(
    amazon_samples["in_response_to_tweet_id"].astype(int)
)

customer_messages = []

with zipfile.ZipFile(ZIP_PATH, "r") as z:
    with z.open("twcs/twcs.csv") as f:
        for chunk in pd.read_csv(f, chunksize=200_000):
            matches = chunk[
                chunk["tweet_id"].isin(reply_ids) &
                (chunk["inbound"] == True)
            ]

            if len(matches):
                customer_messages.append(
                    matches[["tweet_id", "text"]]
                )

customer_messages = pd.concat(customer_messages)

pairs = amazon_samples.merge(
    customer_messages,
    left_on="in_response_to_tweet_id",
    right_on="tweet_id",
    suffixes=("_reply", "_customer")
)

display(
    pairs[["text_customer", "text_reply"]].head(10)
)

,text_customer,text_reply
0,@115850 ... stop cheating... deliver what is o...,@147339 Seems like you had an unpleasant exper...
1,@AmazonHelp I’ve spoken to someone on the phon...,@165000 Sorry but we can't access your account...
2,@AmazonHelp how do i delete my old credit card...,@120404 please check this link here: https://t...
3,@AmazonHelp Every time I can't call to your su...,"@158504 I'm sorry for any trouble, being a soc..."
4,うーんAmazonプライム会員なろっかなぁ,@169659 アマゾンです。Amazonプライムは様々な特典をご用意させていただいておりま...
5,@AmazonHelp ありがとうございます！わかりました！,@142689 どうぞよろしくお願いいたします。\nまた何かご不明な点がございましたらお知ら...
6,@115830 Where is all the PRIME member specific...,@142675 Thanks for reaching out to us! You can...
7,Comprei meus livros na segunda e já chegaram. ...,"@141789 Olá, Amanda! Conte para nós, quais liv..."
8,@AmazonHelp What's the update,"@131026 Hey, this being a social platform we'l..."
9,@149601 @158275 Same Problem. Pre Ordered it 2...,@158274 Have you been in touch with us here: ...


In [9]:
print("AmazonHelp paired conversations:", len(pairs))
print("Customer messages:", pairs["text_customer"].notna().sum())
print("Replies:", pairs["text_reply"].notna().sum())

print(
    "\nAverage customer message length:",
    round(pairs["text_customer"].str.len().mean(), 1)
)

print(
    "Average reply length:",
    round(pairs["text_reply"].str.len().mean(), 1)
)

AmazonHelp paired conversations: 19
Customer messages: 19
Replies: 19

Average customer message length: 90.5
Average reply length: 115.4


## Step 1 — Dataset Analysis & Brand Selection

### Selected Brand: AmazonHelp

AmazonHelp was selected because it provides a large volume of customer-support
interactions with strong reply linkage and diverse support scenarios.

The dataset contains multi-turn customer-support conversations where brand
responses can be linked to customer messages using response tweet IDs.

The paired conversation samples show useful historical resolutions involving
account issues, orders, delivery, Prime, links, and other support requests.

These historical customer → support-response pairs will be used in the next
steps for intent classification, retrieval, grounded response generation,
and escalation decisions.